# Motores de contratos y conciliaciones

Este notebook construye dos tablas listas para los dashboards: `contract_master` (una fila por contrato) y `reconciliation_fact` (una fila por conciliacion). La logica sigue `docs/REGLAS_DASHBOARD.md`.

Los Parquet host se usan con prioridad. Aunque se incorporaron como archivos sin contrato formal, contienen encabezados; se validan por campos requeridos y se normalizan a los nombres documentados.

In [ ]:
from pathlib import Path
from typing import Iterable
import csv
import hashlib
import re
from functools import lru_cache

import pandas as pd

# Parámetros reproducibles. Ajustar AS_OF_DATE para recalcular cortes diarios o mensuales.
ROOT = Path.cwd()
if not (ROOT / 'T_PISD_INSURANCE_CONCILIATION.csv').exists():
    ROOT = ROOT.parent
OUTPUT_DIR = ROOT / 'outputs'
OUTPUT_DIR.mkdir(exist_ok=True)
AS_OF_DATE = pd.Timestamp('2026-09-22').normalize()

ORACLE_CONTRACTS = ROOT / 'T_PISD_INSURANCE_CONTRACT.csv'
ORACLE_CONCILIATIONS = ROOT / 'T_PISD_INSURANCE_CONCILIATION.csv'
ORACLE_MOVEMENTS = ROOT / 'T_PISD_INSR_CONCILIATION_MOV.csv'
HOST_CAP_PARQUET = ROOT / 'PPICH_C_20260922_ICDTCAP.parquet'
HOST_CAM_PARQUET = ROOT / 'PPICH_C_20260922_ICDTCAM.parquet'
HOST_CAP_CSV = ROOT / 'ICDTCAP.csv'
HOST_CAM_CSV = ROOT / 'ICDTCAM.csv'
HOST_LIT_CSV = ROOT / 'ICDTLIT.csv'

VALID_PROCESSES = {'CAN', 'REN', 'INV'}
VALID_CONCILIATION_STATUSES = {'PEN', 'SOL'}
VALID_CONTRACT_STATUSES = {'ANU', 'BAJ', 'ERR', 'FOR', 'PEN', 'INC'}
VALID_RENEWAL_STATUSES = {'ENV', 'NRE', 'REN', 'PRN', 'PRE', 'RZD', 'PNR'}
VALID_ERROR_CODES = {
    'CAN_FILE_NOT_SENT', 'CAN_DUPLICATE_RESP', 'CAN_UNEXPECTED_ERR',
    'CAN_CONCILIATION', 'PRE_NOTFOUND_ERR', 'PRE_AUT_NOTFOUND_ERR',
    'REN_NOTFOUND_ERR', 'REN_AUT_NOTFOUND_ERR',
    'INV_RIMAC_NOT_SENT', 'INV_RIMAC_NOT_CHAR',
}
FREQUENCY_LABELS = {'M': 'Mensual', 'T': 'Trimestral', 'B': 'Bimestral', 'S': 'Semestral', 'A': 'Anual'}
LITERAL_LANGUAGE = 'ES'
DEMO_PRODUCT_WEIGHTS = {'845': 771, '850': 885, '842': 3099, '840': 2781, '832': 3251, '833': 2316, '830': 14252, '834': 6971, '843': 208, '841': 2284, '836': 1421}
quality_rows: list[dict[str, object]] = []

In [ ]:
# Campos minimos de negocio; las columnas host adicionales se perfilan, pero no se usan hasta documentarlas.
CAP_REQUIRED = [
    'CAP-ICCENDIS', 'CAP-ICCOFDIS', 'CAP-ICCD1CTO', 'CAP-ICCD2CTO', 'CAP-ICCCTACT',
    'CAP-ICCCYCIA', 'CAP-ICCPRODC',
    'CAP-ICFALMOV', 'CAP-ICHTIULM', 'CAP-ICFECTE', 'CAP-ICFVENPO',
    'CAP-ICFBAIXA', 'CAP-ICFANCON', 'CAP-ICCMOD01', 'CAP-ICTFOPAG',
    'CAP-ICIPRTOT', 'CAP-ICDIVISA',
]
CAM_REQUIRED = [
    'ICCENDIS', 'ICCOFDIS', 'ICCD1CTO', 'ICCD2CTO', 'ICCCTACT', 'ICNUMOVI',
    'ICFELIQ', 'ICFECOB', 'ICNUMINT', 'ICPRREC', 'ICIPRTOT', 'ICDIVISA', 'ICINDPECO',
]
LIT_REQUIRED = ['ICCCYCIA', 'ICCLITER', 'ICCIOIDI', 'ICLITER1', 'ICHTIULM', 'ICDESLAR']
ORACLE_KEY_COLUMNS = [
    'INSURANCE_CONTRACT_ENTITY_ID', 'INSURANCE_CONTRACT_BRANCH_ID',
    'CONTRACT_FIRST_VERFN_DIGIT_ID', 'CONTRACT_SECOND_VERFN_DIGIT_ID',
    'INSRC_CONTRACT_INT_ACCOUNT_ID',
]

def record_quality(name: str, value: object, status: str = 'info', detail: str = '') -> None:
    quality_rows.append({'check': name, 'value': value, 'status': status, 'detail': detail})

def normalize_component(values: pd.Series, width: int) -> pd.Series:
    """Normaliza una parte de clave sin perder ceros iniciales."""
    normalized = values.astype('string').str.strip()
    normalized = normalized.mask(normalized.isin(['', 'nan', '<NA>']))
    normalized = normalized.str.replace(r'\.0$', '', regex=True)
    return normalized.str.zfill(width)

def build_contract_key(frame: pd.DataFrame, columns: Iterable[str]) -> pd.Series:
    """Devuelve nulo cuando falta cualquiera de los cinco componentes de la clave."""
    parts = frame.loc[:, list(columns)]
    return parts.fillna('').agg(''.join, axis=1).mask(parts.isna().any(axis=1))

def parse_host_date(values: pd.Series) -> pd.Series:
    """Convierte las fechas host y trata 01-01-01 como fecha no informada."""
    raw = values.astype('string').str.strip()
    raw = raw.mask(raw.isin(['', '01-01-01', '0001-01-01', '1900-01-01']))
    return pd.to_datetime(raw, errors='coerce', format='mixed', dayfirst=True).dt.normalize()

def parse_oracle_timestamp(values: pd.Series) -> pd.Series:
    return pd.to_datetime(values, errors='coerce', format='mixed', dayfirst=True)

def numeric(values: pd.Series) -> pd.Series:
    return pd.to_numeric(values.astype('string').str.replace(',', '.', regex=False), errors='coerce')

def assign_demo_product(contract_identifier: object) -> str:
    """Asigna un producto sintético estable según los pesos de distribución de negocio."""
    if pd.isna(contract_identifier):
        return pd.NA
    position = int(hashlib.sha256(str(contract_identifier).encode()).hexdigest()[:16], 16) % sum(DEMO_PRODUCT_WEIGHTS.values())
    cumulative = 0
    for product_code, weight in DEMO_PRODUCT_WEIGHTS.items():
        cumulative += weight
        if position < cumulative:
            return product_code
    raise AssertionError('No se pudo asignar producto de demostración.')

CONTRACT_DECIMAL_COLUMNS = {26, 27, 28, 29, 33, 49, 51, 52, 75, 86, 89}
CONTRACT_TIMESTAMP_COLUMNS = {45, 47, 62, 63}


def read_oracle_contract_csv(path: Path) -> pd.DataFrame:
    """Alinea filas con comas decimales/timestamps no entrecomillados."""
    with path.open(newline='', encoding='utf-8') as source:
        reader = csv.reader(source)
        header = next(reader)
        repaired_rows = []
        repaired_fragments = 0
        for row_number, row in enumerate(reader, start=2):
            @lru_cache(maxsize=None)
            def align(field_index: int, source_index: int):
                if field_index == len(header):
                    return (0, ()) if source_index == len(row) else None
                if source_index >= len(row):
                    return None
                candidates = []
                value = row[source_index]
                next_value = row[source_index + 1] if source_index + 1 < len(row) else None
                timestamp_split = bool(
                    field_index in CONTRACT_TIMESTAMP_COLUMNS and next_value is not None
                    and re.fullmatch(r'\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}', value.strip() or 'x')
                    and re.fullmatch(r'\d{6,9} (AM|PM)', next_value.strip() or 'x')
                )
                decimal_split = bool(
                    field_index in CONTRACT_DECIMAL_COLUMNS and next_value is not None
                    and re.fullmatch(r'-?\d+', value.strip() or 'x')
                    and re.fullmatch(r'\d{1,2}', next_value.strip() or 'x')
                )
                for width, merged_value, bonus in [(1, value, 0)] + ([(2, f'{value},{next_value}', 1)] if timestamp_split or decimal_split else []):
                    remainder = align(field_index + 1, source_index + width)
                    if remainder is None:
                        continue
                    anchor_bonus = 0
                    if field_index == 30:
                        anchor_bonus = 20 if merged_value in {'PEN', 'USD'} else -20
                    elif field_index in {42, 43}:
                        anchor_bonus = 20 if merged_value in VALID_CONTRACT_STATUSES else -20
                    elif field_index in {45, 47}:
                        anchor_bonus = 10 if pd.notna(parse_oracle_timestamp(pd.Series([merged_value])).iloc[0]) else -10
                    candidates.append((remainder[0] + bonus + anchor_bonus, (merged_value,) + remainder[1]))
                return max(candidates, key=lambda candidate: candidate[0]) if candidates else None

            aligned = align(0, 0)
            if aligned is None:
                raise ValueError(f'Fila contractual no alineable: {row_number}, campos={len(row)}')
            repaired_rows.append(aligned[1])
            repaired_fragments += len(row) - len(header)
    record_quality('oracle_contract_repaired_fragments', repaired_fragments, 'warning' if repaired_fragments else 'ok')
    return pd.DataFrame(repaired_rows, columns=header, dtype='string')

def read_oracle_movement_csv(path: Path) -> pd.DataFrame:
    """Alinea importes y timestamps no entrecomillados de movimientos Oracle."""
    with path.open(newline="", encoding="utf-8") as source:
        reader = csv.reader(source)
        header = next(reader)
        repaired_rows = []
        repaired_fragments = 0
        for row_number, row in enumerate(reader, start=2):
            @lru_cache(maxsize=None)
            def align(field_index: int, source_index: int):
                if field_index == len(header):
                    return (0, ()) if source_index == len(row) else None
                if source_index >= len(row):
                    return None
                value = row[source_index]
                next_value = row[source_index + 1] if source_index + 1 < len(row) else None
                timestamp_split = bool(
                    field_index in {11, 13} and next_value is not None
                    and re.fullmatch(r"\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}", value.strip() or "x")
                    and re.fullmatch(r"\d{6,9} (AM|PM)", next_value.strip() or "x")
                )
                decimal_split = bool(
                    field_index in {3, 4} and next_value is not None
                    and re.fullmatch(r"-?\d+", value.strip() or "x")
                    and re.fullmatch(r"\d{1,2}", next_value.strip() or "x")
                )
                candidates = []
                variants = [(1, value, 0)]
                if timestamp_split or decimal_split:
                    variants.append((2, f"{value},{next_value}", 1))
                for width, merged_value, bonus in variants:
                    remainder = align(field_index + 1, source_index + width)
                    if remainder is None:
                        continue
                    anchor_bonus = 0
                    if field_index == 2:
                        anchor_bonus = 20 if merged_value in VALID_PROCESSES else -20
                    elif field_index == 6:
                        anchor_bonus = 10 if "_" in merged_value else -10
                    candidates.append((remainder[0] + bonus + anchor_bonus, (merged_value,) + remainder[1]))
                return max(candidates, key=lambda candidate: candidate[0]) if candidates else None

            aligned = align(0, 0)
            if aligned is None:
                raise ValueError(f"Fila de movimiento no alineable: {row_number}, campos={len(row)}")
            repaired_rows.append(aligned[1])
            repaired_fragments += len(row) - len(header)
    record_quality("oracle_movement_repaired_fragments", repaired_fragments, "warning" if repaired_fragments else "ok")
    return pd.DataFrame(repaired_rows, columns=header, dtype="string")


def read_oracle_movement_csv(path: Path) -> pd.DataFrame:
    """Alinea importes y timestamps no entrecomillados de movimientos Oracle."""
    with path.open(newline="", encoding="utf-8") as source:
        reader = csv.reader(source)
        header = next(reader)
        repaired_rows = []
        repaired_fragments = 0
        for row_number, row in enumerate(reader, start=2):
            @lru_cache(maxsize=None)
            def align(field_index: int, source_index: int):
                if field_index == len(header):
                    return (0, ()) if source_index == len(row) else None
                if source_index >= len(row):
                    return None
                value = row[source_index]
                next_value = row[source_index + 1] if source_index + 1 < len(row) else None
                timestamp_split = bool(
                    field_index in {11, 13} and next_value is not None
                    and re.fullmatch(r"\d{2}/\d{2}/\d{2} \d{2}:\d{2}:\d{2}", value.strip() or "x")
                    and re.fullmatch(r"\d{6,9} (AM|PM)", next_value.strip() or "x")
                )
                decimal_split = bool(
                    field_index in {3, 4} and next_value is not None
                    and re.fullmatch(r"-?\d+", value.strip() or "x")
                    and re.fullmatch(r"\d{1,2}", next_value.strip() or "x")
                )
                candidates = []
                variants = [(1, value, 0)]
                if timestamp_split or decimal_split:
                    variants.append((2, f"{value},{next_value}", 1))
                for width, merged_value, bonus in variants:
                    remainder = align(field_index + 1, source_index + width)
                    if remainder is None:
                        continue
                    anchor_bonus = 0
                    if field_index == 2:
                        anchor_bonus = 20 if merged_value in VALID_PROCESSES else -20
                    elif field_index == 6:
                        anchor_bonus = 10 if "_" in merged_value else -10
                    candidates.append((remainder[0] + bonus + anchor_bonus, (merged_value,) + remainder[1]))
                return max(candidates, key=lambda candidate: candidate[0]) if candidates else None

            aligned = align(0, 0)
            if aligned is None:
                raise ValueError(f"Fila de movimiento no alineable: {row_number}, campos={len(row)}")
            repaired_rows.append(aligned[1])
            repaired_fragments += len(row) - len(header)
    record_quality("oracle_movement_repaired_fragments", repaired_fragments, "warning" if repaired_fragments else "ok")
    return pd.DataFrame(repaired_rows, columns=header, dtype="string")


def read_oracle_csv(path: Path) -> pd.DataFrame:
    # Todas las columnas llegan como texto para proteger identificadores y ceros iniciales.
    if path == ORACLE_CONTRACTS:
        return read_oracle_contract_csv(path)
    if path == ORACLE_MOVEMENTS:
        return read_oracle_movement_csv(path)
    return pd.read_csv(path, dtype='string', keep_default_na=False)

def read_host_cap() -> pd.DataFrame:
    if HOST_CAP_PARQUET.exists():
        parquet_columns = pd.read_parquet(HOST_CAP_PARQUET).columns.tolist()
        required_native = [name.removeprefix('CAP-') for name in CAP_REQUIRED]
        missing = sorted(set(required_native) - set(parquet_columns))
        if missing:
            raise ValueError(f'ICDTCAP Parquet sin columnas requeridas: {missing}')
        extras = sorted(set(parquet_columns) - set(required_native))
        record_quality('icdtcap_extra_columns', len(extras), 'info', ', '.join(extras))
        host = pd.read_parquet(HOST_CAP_PARQUET, columns=required_native)
        return host.rename(columns={name: f'CAP-{name}' for name in required_native})
    host = pd.read_csv(HOST_CAP_CSV, dtype='string', keep_default_na=False)
    missing = sorted(set(CAP_REQUIRED) - set(host.columns))
    if missing:
        raise ValueError(f'ICDTCAP CSV sin columnas requeridas: {missing}')
    return host.loc[:, CAP_REQUIRED].copy()

def read_host_cam() -> pd.DataFrame:
    if HOST_CAM_PARQUET.exists():
        parquet_columns = pd.read_parquet(HOST_CAM_PARQUET).columns.tolist()
        missing = sorted(set(CAM_REQUIRED) - set(parquet_columns))
        if missing:
            raise ValueError(f'ICDTCAM Parquet sin columnas requeridas: {missing}')
        extras = sorted(set(parquet_columns) - set(CAM_REQUIRED))
        record_quality('icdtcam_extra_columns', len(extras), 'info', ', '.join(extras))
        return pd.read_parquet(HOST_CAM_PARQUET, columns=CAM_REQUIRED)
    host = pd.read_csv(HOST_CAM_CSV, dtype='string', keep_default_na=False)
    missing = sorted(set(CAM_REQUIRED) - set(host.columns))
    if missing:
        raise ValueError(f'ICDTCAM CSV sin columnas requeridas: {missing}')
    return host.loc[:, CAM_REQUIRED].copy()

def read_host_lit() -> pd.DataFrame:
    host = pd.read_csv(HOST_LIT_CSV, dtype='string', keep_default_na=False)
    missing = sorted(set(LIT_REQUIRED) - set(host.columns))
    if missing:
        raise ValueError(f'ICDTLIT CSV sin columnas requeridas: {missing}')
    return host.loc[:, LIT_REQUIRED].copy()

In [ ]:
# Ingesta y controles basicos de fuentes.
contracts_raw = read_oracle_csv(ORACLE_CONTRACTS)
conciliations_raw = read_oracle_csv(ORACLE_CONCILIATIONS)
movements_raw = read_oracle_csv(ORACLE_MOVEMENTS)
cap_raw = read_host_cap()
cam_raw = read_host_cam()
literals_raw = read_host_lit()

for source_name, frame in {
    'oracle_contracts': contracts_raw, 'oracle_conciliations': conciliations_raw,
    'oracle_movements': movements_raw, 'icdtcap': cap_raw, 'icdtcam': cam_raw, 'icdtlit': literals_raw,
}.items():
    record_quality(f'{source_name}_rows', len(frame))
    record_quality(f'{source_name}_columns', len(frame.columns))

# Procesos y estados fuera de catalogo se informan antes de construir los outputs.
process_values = set(conciliations_raw['SOURCE_PROCESS_TYPE'].str.strip().dropna())
status_values = set(conciliations_raw['CONCILIATION_STATUS_TYPE'].str.strip().dropna())
record_quality('invalid_processes', len(process_values - VALID_PROCESSES), 'error' if process_values - VALID_PROCESSES else 'ok', str(sorted(process_values - VALID_PROCESSES)))
record_quality('invalid_conciliation_statuses', len(status_values - VALID_CONCILIATION_STATUSES), 'error' if status_values - VALID_CONCILIATION_STATUSES else 'ok', str(sorted(status_values - VALID_CONCILIATION_STATUSES)))

pd.DataFrame(quality_rows).tail(12)

## Motor contractual

ICDTCAP se deduplica por contrato: ultima `CAP-ICHTIULM` y, ante empate, ultima `CAP-ICFALMOV`. El `LEFT JOIN` conserva contratos Oracle que todavia no existen en host.

In [ ]:
# Claves contractuales Oracle. La cuenta se mantiene solo durante la transformacion.
contracts = contracts_raw.copy()
contracts['entity_id'] = normalize_component(contracts['INSURANCE_CONTRACT_ENTITY_ID'], 4)
contracts['branch_id'] = normalize_component(contracts['INSURANCE_CONTRACT_BRANCH_ID'], 4)
contracts['digit_1'] = normalize_component(contracts['CONTRACT_FIRST_VERFN_DIGIT_ID'], 1)
contracts['digit_2'] = normalize_component(contracts['CONTRACT_SECOND_VERFN_DIGIT_ID'], 1)
contracts['account_id'] = normalize_component(contracts['INSRC_CONTRACT_INT_ACCOUNT_ID'], 10)
contracts['contract_identifier'] = build_contract_key(contracts, ['entity_id', 'branch_id', 'digit_1', 'digit_2', 'account_id'])
contracts['contract_lookup_key'] = build_contract_key(contracts, ['entity_id', 'branch_id', 'account_id'])
contracts['creation_date'] = parse_oracle_timestamp(contracts['CREATION_DATE'])

# ICDTCAP tiene nombres normalizados con prefijo CAP- en esta capa.
cap = cap_raw.copy()
cap['entity_id'] = normalize_component(cap['CAP-ICCENDIS'], 4)
cap['branch_id'] = normalize_component(cap['CAP-ICCOFDIS'], 4)
cap['digit_1'] = normalize_component(cap['CAP-ICCD1CTO'], 1)
cap['digit_2'] = normalize_component(cap['CAP-ICCD2CTO'], 1)
cap['account_id'] = normalize_component(cap['CAP-ICCCTACT'], 10)
cap['contract_identifier'] = build_contract_key(cap, ['entity_id', 'branch_id', 'digit_1', 'digit_2', 'account_id'])
cap['literal_company_id'] = normalize_component(cap['CAP-ICCCYCIA'], 2)
cap['source_product_code'] = cap['CAP-ICCPRODC'].astype('string').str.strip()
cap['demo_product_code'] = cap['contract_identifier'].map(assign_demo_product).astype('string')
cap['product_literal_id'] = 'P' + cap['demo_product_code'] + '1'
cap['modality_literal_id'] = cap['demo_product_code'] + cap['CAP-ICCMOD01'].astype('string').str.strip()
cap['CAP-ICHTIULM'] = parse_oracle_timestamp(cap['CAP-ICHTIULM'])
cap['CAP-ICFALMOV'] = parse_host_date(cap['CAP-ICFALMOV'])
for date_column in ['CAP-ICFECTE', 'CAP-ICFVENPO', 'CAP-ICFBAIXA', 'CAP-ICFANCON']:
    cap[date_column] = parse_host_date(cap[date_column])
cap['CAP-ICIPRTOT'] = numeric(cap['CAP-ICIPRTOT'])

cap_duplicates = cap['contract_identifier'].duplicated(keep=False).sum()
record_quality('icdtcap_duplicate_contract_rows', int(cap_duplicates))
cap_latest = (
    cap.sort_values(['CAP-ICHTIULM', 'CAP-ICFALMOV'], ascending=[False, False], na_position='last')
       .drop_duplicates('contract_identifier', keep='first')
)

literals = literals_raw.copy()
literals['literal_company_id'] = normalize_component(literals['ICCCYCIA'], 2)
literals['literal_id'] = literals['ICCLITER'].astype('string').str.strip()
literals['language_id'] = literals['ICCIOIDI'].astype('string').str.strip()
literals['audit_date'] = parse_oracle_timestamp(literals['ICHTIULM'])
literals = (
    literals.loc[literals['language_id'].eq(LITERAL_LANGUAGE)]
            .sort_values('audit_date', ascending=False, na_position='last')
            .drop_duplicates(['literal_company_id', 'literal_id'], keep='first')
)
product_literals = literals[['literal_company_id', 'literal_id', 'ICLITER1']].rename(columns={'literal_id': 'product_literal_id', 'ICLITER1': 'product_name'})
modality_literals = literals[['literal_company_id', 'literal_id', 'ICLITER1']].rename(columns={'literal_id': 'modality_literal_id', 'ICLITER1': 'modality_name'})
cap_latest = (
    cap_latest.merge(product_literals, on=['literal_company_id', 'product_literal_id'], how='left', validate='many_to_one')
              .merge(modality_literals, on=['literal_company_id', 'modality_literal_id'], how='left', validate='many_to_one')
)
record_quality('icdtlit_product_name_missing', int(cap_latest['product_name'].isna().sum()), 'warning' if cap_latest['product_name'].isna().any() else 'ok')
record_quality('icdtlit_modality_name_missing', int(cap_latest['modality_name'].isna().sum()), 'warning' if cap_latest['modality_name'].isna().any() else 'ok')

if contracts['contract_identifier'].duplicated().any():
    raise ValueError('La fuente contractual Oracle no tiene grano unico por contrato.')

contract_master_internal = contracts.merge(
    cap_latest[[
        'contract_identifier', 'CAP-ICFECTE', 'CAP-ICFVENPO', 'CAP-ICFBAIXA',
        'CAP-ICFANCON', 'CAP-ICCMOD01', 'CAP-ICTFOPAG', 'CAP-ICIPRTOT', 'CAP-ICDIVISA',
        'demo_product_code', 'product_name', 'modality_name',
    ]],
    on='contract_identifier', how='left', validate='one_to_one', indicator='icdtcap_join',
)
contract_master_internal['payment_frequency_code'] = contract_master_internal['CAP-ICTFOPAG'].astype('string').str.strip()
contract_master_internal['payment_frequency'] = contract_master_internal['payment_frequency_code'].map(FREQUENCY_LABELS).fillna('Sin catalogar')
contract_master_internal['display_product_id'] = contract_master_internal['demo_product_code'].fillna(contract_master_internal['INSURANCE_PRODUCT_ID'])
contract_master_internal['cancellation_date'] = contract_master_internal['CAP-ICFBAIXA'].where(
    contract_master_internal['CONTRACT_STATUS_ID'].eq('BAJ'),
    contract_master_internal['CAP-ICFANCON'].where(contract_master_internal['CONTRACT_STATUS_ID'].eq('ANU')),
)
contract_master_internal['days_to_expiry'] = (contract_master_internal['CAP-ICFVENPO'] - AS_OF_DATE).dt.days
contract_master_internal['renewal_confirmation_due_date'] = contract_master_internal['CAP-ICFVENPO'] - pd.Timedelta(days=50)
contract_master_internal['renewal_receipts_due_date'] = contract_master_internal['CAP-ICFVENPO'] - pd.Timedelta(days=45)
contract_master_internal['icdtcap_matched'] = contract_master_internal['icdtcap_join'].eq('both')
record_quality('contracts_without_icdtcap', int((~contract_master_internal['icdtcap_matched']).sum()))

contract_master = contract_master_internal.rename(columns={
    'POLICY_ID': 'policy_id', 'CUSTOMER_ID': 'customer_id',
    'INSURANCE_PRODUCT_ID': 'insurance_product_id',
    'INSRNC_CO_CONTRACT_STATUS_TYPE': 'rimac_contract_status',
    'CONTRACT_STATUS_ID': 'bank_contract_status',
    'CONTRACT_RENEWAL_STATUS_TYPE': 'renewal_status',
    'SOURCE_PLATFORM_TYPE': 'source_platform_type',
    'CAP-ICCMOD01': 'insurance_modality', 'CAP-ICIPRTOT': 'periodic_premium',
    'CAP-ICDIVISA': 'premium_currency', 'CAP-ICFECTE': 'coverage_start_date',
    'CAP-ICFVENPO': 'coverage_end_date',
})[[
    'contract_identifier', 'policy_id', 'customer_id', 'insurance_product_id',
    'rimac_contract_status', 'bank_contract_status', 'creation_date', 'renewal_status',
    'source_platform_type', 'insurance_modality', 'payment_frequency_code',
    'payment_frequency', 'display_product_id', 'product_name', 'modality_name', 'periodic_premium', 'premium_currency', 'coverage_start_date',
    'coverage_end_date', 'cancellation_date', 'days_to_expiry',
    'renewal_confirmation_due_date', 'renewal_receipts_due_date', 'icdtcap_matched',
]]
contract_master.head()

In [ ]:
# Control de calidad del catálogo de renovación confirmado por negocio.
renewal_statuses_found = set(contract_master['renewal_status'].dropna().astype('string').str.strip())
invalid_renewal_statuses = renewal_statuses_found - VALID_RENEWAL_STATUSES
record_quality(
    'invalid_renewal_statuses',
    len(invalid_renewal_statuses),
    'error' if invalid_renewal_statuses else 'ok',
    str(sorted(invalid_renewal_statuses)),
)

## Motor de conciliaciones

Los movimientos Oracle se agregan exclusivamente por `CONCILIATION_SEQUENTIAL_ID`. ICDTCAM se agrega por contrato, porque las fuentes no proveen una clave documentada que relacione un recibo host con una conciliacion individual.

In [ ]:
# Agregado de movimientos Oracle. Los importes no aplican a CAN ni REN.
movements = movements_raw.copy()
movements['CONCILIATION_SEQUENTIAL_ID'] = movements['CONCILIATION_SEQUENTIAL_ID'].astype('string').str.strip()
movements['PAYMENT_AMOUNT'] = numeric(movements['PAYMENT_AMOUNT'])
movements['OPERATED_AMOUNT'] = numeric(movements['OPERATED_AMOUNT'])
movement_agg = movements.groupby('CONCILIATION_SEQUENTIAL_ID', dropna=False).agg(
    oracle_movement_count=('OPER_SEQ_MOV_ID', 'size'),
    payment_amount=('PAYMENT_AMOUNT', 'sum'),
    operated_amount=('OPERATED_AMOUNT', 'sum'),
).reset_index()

# ICDTCAM se conserva a grano de movimiento hasta este agregado contractual.
cam = cam_raw.copy()
cam['entity_id'] = normalize_component(cam['ICCENDIS'], 4)
cam['branch_id'] = normalize_component(cam['ICCOFDIS'], 4)
cam['digit_1'] = normalize_component(cam['ICCD1CTO'], 1)
cam['digit_2'] = normalize_component(cam['ICCD2CTO'], 1)
cam['account_id'] = normalize_component(cam['ICCCTACT'], 10)
cam['contract_identifier'] = build_contract_key(cam, ['entity_id', 'branch_id', 'digit_1', 'digit_2', 'account_id'])
cam['ICFELIQ'] = parse_host_date(cam['ICFELIQ'])
cam['ICFECOB'] = parse_host_date(cam['ICFECOB'])
cam['ICNUMINT'] = numeric(cam['ICNUMINT']).fillna(0)
cam['ICPRREC'] = numeric(cam['ICPRREC'])
cam['ICIPRTOT'] = numeric(cam['ICIPRTOT'])
cam['is_collected'] = cam['ICFECOB'].notna()
cam['is_collected_today'] = cam['ICFECOB'].eq(AS_OF_DATE)
cam['is_collected_month'] = cam['ICFECOB'].dt.to_period('M').eq(AS_OF_DATE.to_period('M'))
cam['is_attempted_today'] = cam['ICFELIQ'].eq(AS_OF_DATE) & cam['ICNUMINT'].gt(0)
cam['is_effective_today'] = cam['is_attempted_today'] & cam['is_collected_today']

cam_dashboard = cam.merge(
    contract_master[['contract_identifier', 'policy_id', 'insurance_product_id', 'display_product_id', 'product_name', 'insurance_modality', 'modality_name', 'payment_frequency_code', 'payment_frequency']],
    on='contract_identifier', how='left', validate='many_to_one', indicator='contract_join',
).rename(columns={
    'ICNUMOVI': 'receipt_movement_number', 'ICFELIQ': 'settlement_date',
    'ICFECOB': 'collection_date', 'ICNUMINT': 'charge_attempts',
    'ICIPRTOT': 'invoiced_premium_amount', 'ICPRREC': 'net_collected_amount',
    'ICDIVISA': 'currency', 'ICINDPECO': 'receipt_collection_status',
})
record_quality('cam_receipts_without_contract_match', int(cam_dashboard['contract_join'].ne('both').sum()))

cam_agg = cam.groupby('contract_identifier', dropna=False).agg(
    host_receipt_count=('ICNUMOVI', 'size'),
    host_collected_receipt_count=('is_collected', 'sum'),
    host_collected_today_count=('is_collected_today', 'sum'),
    host_collected_month_count=('is_collected_month', 'sum'),
    host_attempted_today_count=('is_attempted_today', 'sum'),
    host_effective_today_count=('is_effective_today', 'sum'),
    host_attempts_today=('ICNUMINT', lambda values: values[cam.loc[values.index, 'is_attempted_today']].sum()),
    host_net_collected_amount=('ICPRREC', 'sum'),
).reset_index()
cam_agg['host_daily_collection_effectiveness'] = (
    cam_agg['host_effective_today_count'] / cam_agg['host_attempted_today_count'].replace(0, pd.NA)
)

# La conciliacion no contiene digitos verificadores: solo se une si entidad+oficina+cuenta identifica un contrato sin ambiguedad.
lookup_counts = contract_master_internal.groupby('contract_lookup_key')['contract_identifier'].nunique()
ambiguous_lookup_keys = set(lookup_counts[lookup_counts.gt(1)].index)
record_quality('ambiguous_conciliation_contract_keys', len(ambiguous_lookup_keys), 'warning' if ambiguous_lookup_keys else 'ok')
contract_lookup = contract_master_internal.loc[
    ~contract_master_internal['contract_lookup_key'].isin(ambiguous_lookup_keys),
    ['contract_lookup_key', 'contract_identifier', 'POLICY_ID', 'INSURANCE_PRODUCT_ID',
     'CAP-ICCMOD01', 'payment_frequency_code', 'payment_frequency', 'display_product_id', 'product_name', 'modality_name'],
].rename(columns={'POLICY_ID': 'contract_policy_id'})

reconciliation = conciliations_raw.copy()
reconciliation['CONCILIATION_SEQUENTIAL_ID'] = reconciliation['CONCILIATION_SEQUENTIAL_ID'].astype('string').str.strip()
reconciliation['entity_id'] = normalize_component(reconciliation['INSURANCE_CONTRACT_ENTITY_ID'], 4)
reconciliation['branch_id'] = normalize_component(reconciliation['INSURANCE_CONTRACT_BRANCH_ID'], 4)
reconciliation['account_id'] = normalize_component(reconciliation['INSRC_CONTRACT_INT_ACCOUNT_ID'], 10)
reconciliation['contract_lookup_key'] = build_contract_key(reconciliation, ['entity_id', 'branch_id', 'account_id'])
reconciliation['creation_date'] = parse_oracle_timestamp(reconciliation['CREATION_DATE'])
reconciliation['audit_date'] = parse_oracle_timestamp(reconciliation['AUDIT_DATE'])

conciliation_duplicates = reconciliation['CONCILIATION_SEQUENTIAL_ID'].duplicated(keep=False).sum()
record_quality('duplicate_conciliation_source_rows', int(conciliation_duplicates), 'warning' if conciliation_duplicates else 'ok')
reconciliation = (
    reconciliation.sort_values(['audit_date', 'creation_date'], ascending=[False, False], na_position='last')
                  .drop_duplicates('CONCILIATION_SEQUENTIAL_ID', keep='first')
)

reconciliation_internal = (
    reconciliation.merge(movement_agg, on='CONCILIATION_SEQUENTIAL_ID', how='left', validate='one_to_one')
                  .merge(contract_lookup, on='contract_lookup_key', how='left', validate='many_to_one')
                  .merge(cam_agg, on='contract_identifier', how='left', validate='many_to_one')
)
is_inv = reconciliation_internal['SOURCE_PROCESS_TYPE'].eq('INV')
for column in ['payment_amount', 'operated_amount']:
    reconciliation_internal[column] = reconciliation_internal[column].where(is_inv)
reconciliation_internal['amount_difference'] = reconciliation_internal['payment_amount'] - reconciliation_internal['operated_amount']
reconciliation_internal['days_open'] = (AS_OF_DATE - reconciliation_internal['creation_date'].dt.normalize()).dt.days
reconciliation_internal['process_label'] = reconciliation_internal['SOURCE_PROCESS_TYPE'].map({
    'CAN': 'Cancelacion', 'REN': 'Renovacion', 'INV': 'Cobranza',
})
reconciliation_internal['renewal_incidence_type'] = pd.NA
reconciliation_internal.loc[reconciliation_internal['ERROR_CODE_ID'].isin(['PRE_NOTFOUND_ERR', 'PRE_AUT_NOTFOUND_ERR']), 'renewal_incidence_type'] = 'Confirmacion pendiente'
reconciliation_internal.loc[reconciliation_internal['ERROR_CODE_ID'].isin(['REN_NOTFOUND_ERR', 'REN_AUT_NOTFOUND_ERR']), 'renewal_incidence_type'] = 'Recibos pendientes'
reconciliation_internal['rimac_collection_result'] = pd.NA
reconciliation_internal.loc[
    is_inv & reconciliation_internal['CONCILIATION_STATUS_TYPE'].eq('PEN') & reconciliation_internal['ERROR_CODE_ID'].eq('INV_RIMAC_NOT_SENT'),
    'rimac_collection_result',
] = 'Cobro automatico realizado no informado'
reconciliation_internal.loc[
    is_inv & reconciliation_internal['CONCILIATION_STATUS_TYPE'].eq('PEN') & reconciliation_internal['ERROR_CODE_ID'].eq('INV_RIMAC_NOT_CHAR'),
    'rimac_collection_result',
] = 'No cobrado ni informado'

reconciliation_fact = reconciliation_internal.rename(columns={
    'CONCILIATION_SEQUENTIAL_ID': 'conciliation_id', 'POLICY_RECEIPT_ID': 'policy_receipt_id',
    'SOURCE_PROCESS_TYPE': 'process_type', 'ERROR_CODE_ID': 'error_code',
    'ERROR_CODE_DESC': 'error_description', 'CONCILIATION_STATUS_TYPE': 'conciliation_status',
    'contract_policy_id': 'policy_id', 'INSURANCE_PRODUCT_ID': 'insurance_product_id',
    'CAP-ICCMOD01': 'insurance_modality',
})[[
    'conciliation_id', 'contract_identifier', 'policy_id', 'policy_receipt_id',
    'insurance_product_id', 'display_product_id', 'product_name', 'insurance_modality', 'modality_name', 'payment_frequency_code',
    'payment_frequency', 'process_type', 'process_label', 'error_code',
    'error_description', 'conciliation_status', 'creation_date', 'audit_date',
    'days_open', 'oracle_movement_count', 'payment_amount', 'operated_amount',
    'amount_difference', 'host_receipt_count', 'host_collected_receipt_count',
    'host_collected_today_count', 'host_collected_month_count',
    'host_attempted_today_count', 'host_effective_today_count', 'host_attempts_today',
    'host_daily_collection_effectiveness', 'host_net_collected_amount',
    'renewal_incidence_type', 'rimac_collection_result',
]]
reconciliation_fact.head()

In [ ]:
# Validaciones de publicacion y escritura de las dos tablas finales.
if contract_master['contract_identifier'].duplicated().any():
    raise AssertionError('contract_master no es unico por contrato.')
if reconciliation_fact['conciliation_id'].duplicated().any():
    raise AssertionError('reconciliation_fact no es unico por conciliacion.')
if any('INSRC_CONTRACT_INT_ACCOUNT_ID' in frame.columns for frame in [contract_master, reconciliation_fact]):
    raise AssertionError('La cuenta interna no puede existir en los outputs.')
if reconciliation_fact.loc[reconciliation_fact['process_type'].ne('INV'), ['payment_amount', 'operated_amount']].notna().any().any():
    raise AssertionError('CAN y REN no pueden exponer importes.')

record_quality('contract_master_rows', len(contract_master), 'ok')
record_quality('reconciliation_fact_rows', len(reconciliation_fact), 'ok')
record_quality('conciliation_without_contract_match', int(reconciliation_fact['contract_identifier'].isna().sum()))
record_quality('invalid_error_codes', int((~reconciliation_fact['error_code'].isin(VALID_ERROR_CODES)).sum()))

contract_master.to_csv(OUTPUT_DIR / 'contract_master.csv', index=False, date_format='%Y-%m-%d')
reconciliation_fact.to_csv(OUTPUT_DIR / 'reconciliation_fact.csv', index=False, date_format='%Y-%m-%d')
quality_report = pd.DataFrame(quality_rows)
quality_report.to_csv(OUTPUT_DIR / 'quality_report.csv', index=False)

display(quality_report)
print(f'contract_master: {len(contract_master):,} filas -> {OUTPUT_DIR / "contract_master.csv"}')
print(f'reconciliation_fact: {len(reconciliation_fact):,} filas -> {OUTPUT_DIR / "reconciliation_fact.csv"}')

## Limites documentados


- ICDTCAM conserva su grano de movimiento hasta el agregado por contrato; no se crea una relacion recibo-conciliacion no demostrable.


- ICDTLIT se incorpora como catálogo sintético temporal; sus productos ponderados y sus modalidades secuenciales deben sustituirse por el catálogo real antes de producción.


- ICDTCAN tiene una plantilla y diccionario, pero no participa hasta recibir registros y confirmar granularidad, reversiones y precedencia de fechas. ICDTPROD no se crea porque no hay regla vigente ni diccionario que sustente su estructura.


- Las columnas host adicionales se registran en `quality_report.csv`; deben documentarse antes de incorporarse a una metrica o join.

# Dashboard operativo

Overview, Alta, Cancelación y Renovación consumen `contract_master` y `reconciliation_fact`. Cobranza consume `cam_dashboard` directamente a grano de recibo y usa las conciliaciones `INV` solo para sus alertas de información RIMAC. Los filtros se definen en la siguiente celda; las métricas sin una fuente demostrable se presentan como **TBD**.

In [ ]:
import matplotlib.pyplot as plt
from IPython.display import display, Markdown

# Parámetros de visualización. Cambiar estos valores vuelve a calcular las cinco secciones.
DASHBOARD_START = pd.Timestamp('2026-01-01')
DASHBOARD_END = AS_OF_DATE
DASHBOARD_PRODUCTS: list[str] = []
DASHBOARD_FREQUENCIES: list[str] = []

plt.rcParams.update({'figure.figsize': (10, 4), 'axes.grid': True, 'grid.alpha': 0.2})


def apply_filters(frame: pd.DataFrame, date_column: str, products: list[str] | None = None, frequencies: list[str] | None = None) -> pd.DataFrame:
    """Aplica periodo, producto y frecuencia cuando esas columnas están disponibles."""
    result = frame.copy()
    if date_column in result:
        result = result.loc[result[date_column].between(DASHBOARD_START, DASHBOARD_END, inclusive='both')]
    if products and 'insurance_product_id' in result:
        result = result.loc[result['insurance_product_id'].astype('string').isin(products)]
    if frequencies and 'payment_frequency' in result:
        result = result.loc[result['payment_frequency'].isin(frequencies)]
    return result


def show_kpis(title: str, values: dict[str, object]) -> None:
    display(Markdown(f'### {title}'))
    display(pd.DataFrame([values]).T.rename(columns={0: 'Valor'}))


def plot_counts(series: pd.Series, title: str, kind: str = 'bar') -> None:
    if series.empty:
        display(Markdown(f'**TBD / sin datos:** {title}'))
        return
    axis = series.plot(kind=kind, color='#147d9a', title=title)
    axis.set_xlabel('')
    axis.set_ylabel('Casos')
    plt.tight_layout()
    plt.show()


def top_products(frame: pd.DataFrame, limit: int = 10) -> pd.Series:
    if 'product_name' in frame and frame['product_name'].notna().any():
        labels = frame['product_name'].fillna(frame['insurance_product_id'].astype('string'))
    else:
        labels = frame['insurance_product_id'].astype('string')
    return labels.fillna('Sin match').value_counts().head(limit)


def display_detail(frame: pd.DataFrame, columns: list[str], limit: int = 25) -> None:
    available = [column for column in columns if column in frame.columns]
    display(frame.loc[:, available].head(limit))

## 1. Overview

Visión ejecutiva del período. Las aperturas y altas responden a los filtros definidos arriba.

In [ ]:
overview_contracts = apply_filters(contract_master, 'creation_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
overview_reconciliation = apply_filters(reconciliation_fact, 'creation_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
show_kpis('KPIs de Overview', {
    'Altas en período': len(overview_contracts),
    'CAN abiertas': int(((overview_reconciliation['process_type'] == 'CAN') & (overview_reconciliation['conciliation_status'] == 'PEN')).sum()),
    'REN abiertas': int(((overview_reconciliation['process_type'] == 'REN') & (overview_reconciliation['conciliation_status'] == 'PEN')).sum()),
    'INV abiertas': int(((overview_reconciliation['process_type'] == 'INV') & (overview_reconciliation['conciliation_status'] == 'PEN')).sum()),
})
plot_counts(overview_reconciliation['process_label'].value_counts(), 'Aperturas de conciliación por proceso')
plot_counts(top_products(pd.concat([overview_contracts, overview_reconciliation], ignore_index=True)), 'Top 10 productos del período')
display(overview_reconciliation.groupby(['process_label', 'conciliation_status'], dropna=False).size().rename('casos').reset_index())

## 2. Altas

Construido desde `contract_master`; la fecha de alta es `creation_date` y el inicio de cobertura se muestra aparte.

In [ ]:
new_signups = apply_filters(contract_master, 'creation_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
historical_pending = contract_master.loc[
    contract_master['rimac_contract_status'].eq('PEN')
    & (~DASHBOARD_PRODUCTS or contract_master['insurance_product_id'].astype('string').isin(DASHBOARD_PRODUCTS))
    & (~DASHBOARD_FREQUENCIES or contract_master['payment_frequency'].isin(DASHBOARD_FREQUENCIES))
]
formalized = new_signups['rimac_contract_status'].eq('FOR')
show_kpis('KPIs de Altas', {
    'Altas en período': len(new_signups),
    'Altas hoy': int(new_signups['creation_date'].dt.normalize().eq(AS_OF_DATE).sum()),
    'Altas del mes': int(new_signups['creation_date'].dt.to_period('M').eq(AS_OF_DATE.to_period('M')).sum()),
    'Pendientes históricos RIMAC': len(historical_pending),
    'Formalizadas del período': int(formalized.sum()),
    'Tasa de formalización': round(100 * formalized.mean(), 2) if len(new_signups) else pd.NA,
})
plot_counts(new_signups.set_index('creation_date').resample('ME').size(), 'Altas por mes', kind='line')
plot_counts(top_products(new_signups), 'Altas por producto')
display_detail(new_signups, ['policy_id', 'insurance_product_id', 'rimac_contract_status', 'payment_frequency', 'insurance_modality', 'creation_date', 'coverage_start_date', 'periodic_premium', 'premium_currency'])

## 3. Cancelaciones

El corte fijo viene de contratos cancelados; las incidencias y resoluciones filtradas vienen de conciliaciones `CAN`.

In [ ]:
cancellation_fixed = contract_master.loc[contract_master['bank_contract_status'].isin(['BAJ', 'ANU'])]
cancellation_cases = apply_filters(reconciliation_fact.loc[reconciliation_fact['process_type'].eq('CAN')], 'creation_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
resolved = cancellation_cases['conciliation_status'].eq('SOL')
resolution_days = (cancellation_cases.loc[resolved, 'audit_date'].dt.normalize() - cancellation_cases.loc[resolved, 'creation_date'].dt.normalize()).dt.days
show_kpis('Corte fijo de Cancelaciones', {
    'Cancelados hoy': int(cancellation_fixed['cancellation_date'].eq(AS_OF_DATE).sum()),
    'Cancelados del mes': int(cancellation_fixed['cancellation_date'].dt.to_period('M').eq(AS_OF_DATE.to_period('M')).sum()),
})
show_kpis('Resultado filtrado de Cancelaciones', {
    'Conciliaciones abiertas': int(cancellation_cases['conciliation_status'].eq('PEN').sum()),
    'Abiertos sin cancelar en RIMAC': int((cancellation_cases['conciliation_status'].eq('PEN') & cancellation_cases['error_code'].eq('CAN_FILE_NOT_SENT')).sum()),
    'Tasa de resolución': round(100 * resolved.mean(), 2) if len(cancellation_cases) else pd.NA,
    'Tiempo medio de resolución (días)': round(resolution_days.mean(), 2),
})
plot_counts(cancellation_cases.loc[cancellation_cases['conciliation_status'].eq('PEN'), 'error_code'].value_counts(), 'Motivos pendientes de cancelación')
plot_counts(cancellation_cases.set_index('creation_date').resample('ME').size(), 'Aperturas de cancelación por mes', kind='line')
display_detail(cancellation_cases, ['conciliation_id', 'policy_id', 'insurance_product_id', 'insurance_modality', 'error_code', 'conciliation_status', 'creation_date', 'audit_date', 'days_open'])

## 4. Renovaciones

La base contractual usa fin de vigencia y los casos abiertos `REN` expresan la información faltante; no se infiere una no renovación sin estado `NRE`.

In [ ]:
renewal_contracts = apply_filters(contract_master, 'coverage_end_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
renewal_contracts = renewal_contracts.loc[~renewal_contracts['bank_contract_status'].isin(['BAJ', 'ANU', 'ERR'])]
renewal_cases = apply_filters(reconciliation_fact.loc[reconciliation_fact['process_type'].eq('REN')], 'creation_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
open_renewal_cases = renewal_cases.loc[renewal_cases['conciliation_status'].eq('PEN')]
incidence_contracts = set(open_renewal_cases['contract_identifier'].dropna())
confirmation_pending = open_renewal_cases['error_code'].isin(['PRE_NOTFOUND_ERR', 'PRE_AUT_NOTFOUND_ERR'])
receipts_pending = open_renewal_cases['error_code'].isin(['REN_NOTFOUND_ERR', 'REN_AUT_NOTFOUND_ERR'])
renewed = renewal_contracts['renewal_status'].eq('REN')
rejected = renewal_contracts['renewal_status'].eq('NRE')
show_kpis('Corte fijo de Renovaciones', {
    'Renovadas hoy': int((contract_master['renewal_status'].eq('REN') & contract_master['coverage_start_date'].eq(AS_OF_DATE)).sum()),
    'Renovadas del mes': int((contract_master['renewal_status'].eq('REN') & contract_master['coverage_end_date'].dt.to_period('M').eq(AS_OF_DATE.to_period('M'))).sum()),
})
show_kpis('Resultado filtrado de Renovaciones', {
    'Pólizas previstas para renovar': len(renewal_contracts),
    'Confirmación pendiente': int(confirmation_pending.sum()),
    'Recibos pendientes': int(receipts_pending.sum()),
    'Por renovar con información pendiente': int(renewal_contracts['contract_identifier'].isin(incidence_contracts).sum()),
    'Renovadas': int(renewed.sum()),
    'No renovación por decisión RIMAC': int(rejected.sum()),
})
plot_counts(renewal_contracts['renewal_status'].fillna('Sin estado').value_counts(), 'Estados de renovación del período')
plot_counts(open_renewal_cases['renewal_incidence_type'].value_counts(), 'Incidencias abiertas de renovación')
plot_counts(top_products(renewal_cases), 'Incidencias de renovación por producto')
renewal_detail = renewal_contracts.merge(open_renewal_cases[['contract_identifier', 'renewal_incidence_type', 'error_code', 'conciliation_status']], on='contract_identifier', how='left', validate='one_to_many')
display_detail(renewal_detail, ['policy_id', 'insurance_product_id', 'insurance_modality', 'payment_frequency', 'coverage_end_date', 'days_to_expiry', 'renewal_status', 'renewal_incidence_type', 'error_code'])

## 5. Cobranza

Los importes y casos `INV` se calculan desde `reconciliation_fact`. El corte global de recibos queda como `TBD` hasta que el motor publique el agregado ICDTCAM completo.

In [ ]:
collection_receipts = apply_filters(cam_dashboard, 'settlement_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
collected_receipts = apply_filters(cam_dashboard, 'collection_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
collection_cases = apply_filters(reconciliation_fact.loc[reconciliation_fact['process_type'].eq('INV')], 'creation_date', DASHBOARD_PRODUCTS, DASHBOARD_FREQUENCIES)
open_collection_cases = collection_cases['conciliation_status'].eq('PEN')
automatic_not_reported = open_collection_cases & collection_cases['error_code'].eq('INV_RIMAC_NOT_SENT')
not_collected_not_reported = open_collection_cases & collection_cases['error_code'].eq('INV_RIMAC_NOT_CHAR')
show_kpis('Corte fijo de Cobranza', {
    'Recibos cobrados': int(cam_dashboard['is_collected'].sum()),
    'Cobrados hoy': int(cam_dashboard['is_collected_today'].sum()),
    'Cobrados del mes': int(cam_dashboard['is_collected_month'].sum()),
    'Intentados hoy': int(cam_dashboard['is_attempted_today'].sum()),
    'Efectividad diaria': round(100 * cam_dashboard['is_effective_today'].sum() / cam_dashboard['is_attempted_today'].sum(), 2) if cam_dashboard['is_attempted_today'].any() else pd.NA,
})
show_kpis('Resultado filtrado de Cobranza', {
    'Recibos a liquidar': len(collection_receipts),
    'Importe facturado': round(collection_receipts['invoiced_premium_amount'].sum(min_count=1), 2),
    'Recibos cobrados en período': len(collected_receipts),
    'Monto cobrado en período': round(collected_receipts['net_collected_amount'].sum(min_count=1), 2),
    'Conciliaciones abiertas': int(open_collection_cases.sum()),
    'Cobro automático no informado': int(automatic_not_reported.sum()),
    'No cobrado ni informado': int(not_collected_not_reported.sum()),
})
plot_counts(collection_receipts['receipt_collection_status'].value_counts(), 'Situación de cobro de recibos a liquidar')
plot_counts(top_products(collection_receipts), 'Recibos a liquidar por producto')
display_detail(collection_receipts, ['policy_id', 'receipt_movement_number', 'insurance_product_id', 'insurance_modality', 'payment_frequency', 'settlement_date', 'collection_date', 'charge_attempts', 'invoiced_premium_amount', 'net_collected_amount', 'currency', 'receipt_collection_status'])